# 24 FAST — Dua dataset dan keluarga Holm

Jalankan sesudah notebook 22 FAST dan 23 FAST lengkap. Tabel hasil nyata tetap
memiliki delapan strata. Tidak menjalankan model, training atau intervensi.
Notebook dan seluruh hasil lama dipertahankan.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
from contextlib import contextmanager
from importlib.metadata import version
import csv, gzip, json, os, pickle, platform, socket, sys, time
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.spatial.distance import jensenshannon
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/'src/e2_checkpoints.py').is_file())
sys.path.insert(0, str(ROOT/'src'))
from audit_e0_e1 import AlignmentError, cache_manifest, file_sha256
from e2_checkpoints import spec_digest
PLAN = ROOT/'plans/attention_audit_replication20_20261019'
LOCK = json.loads((PLAN/'design_lock.json').read_text())
C = LOCK['configuration']
if file_sha256(PLAN/'PROTOCOL.md') != LOCK['protocol_sha256']:
    raise AlignmentError('protocol changed after design lock; rebuild before new outcomes')
for relative, expected in LOCK['inputs'].items():
    if file_sha256(ROOT/relative) != expected:
        raise AlignmentError('pinned input/source changed: '+relative)
if file_sha256(PLAN/'directed_seed_schedule.csv') != LOCK['schedule_sha256']:
    raise AlignmentError('transfer schedule differs from design lock')
schedule_all = pd.read_csv(PLAN/'directed_seed_schedule.csv')
schedule_all['primary_pair'] = schedule_all.primary_pair.astype(str).str.lower().map({'true': True, 'false': False})
if schedule_all.primary_pair.isna().any(): raise AlignmentError('invalid schedule boolean')


def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)


def atomic_pickle(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    blob=gzip.compress(pickle.dumps(value,protocol=5),compresslevel=3)
    with tmp.open('wb') as stream:pickle.dump(dict(format='gzip_pickle_v1',sha256=sha256(blob).hexdigest(),payload=blob),stream,protocol=5)
    os.replace(tmp, path)


def read_local_pickle(path):
    with Path(path).open('rb') as stream:envelope=pickle.load(stream)
    if envelope['format']!='gzip_pickle_v1' or sha256(envelope['payload']).hexdigest()!=envelope['sha256']:
        raise AlignmentError('cached payload checksum mismatch: '+str(path))
    return pickle.loads(gzip.decompress(envelope['payload']))


def notebook_code_digest(path):
    doc=json.loads(Path(path).read_text())
    chunks=[]
    for cell in doc['cells']:
        if cell['cell_type']=='code':
            value=cell['source'];chunks.append(''.join(value) if isinstance(value,list) else value)
    return sha256('\n'.join(chunks).encode()).hexdigest()


def atomic_csv(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    data.to_csv(tmp, index=False); os.replace(tmp, path)


@contextmanager
def stage_lock(directory, stage):
    path = directory/(stage+'.active.lock')
    directory.mkdir(parents=True, exist_ok=True)
    if path.exists():
        owner = json.loads(path.read_text())
        if owner['host'] != socket.gethostname():
            raise RuntimeError('stage lock belongs to another host; inspect before recovering: '+str(path))
        try: os.kill(int(owner['pid']), 0)
        except ProcessLookupError:
            path.rename(path.with_name(path.name+'.stale.'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')))
        else:
            raise RuntimeError('stage already running in PID '+str(owner['pid'])+'; no concurrent Run All')
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL)
    with os.fdopen(fd, 'w') as stream:
        json.dump(dict(pid=os.getpid(), host=socket.gethostname()), stream)
    try: yield
    except BaseException as error:
        if 'manifest' in globals() and 'OUT' in globals():
            manifest.update(completion_status='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',
                            failed_stage=stage,error_type=type(error).__name__,error_message=str(error))
            atomic_json(OUT/'analysis_manifest.json',manifest)
        raise
    finally:
        if path.exists(): path.unlink()


def prepare_cohort(cohort):
    cache = ROOT/C[cohort+'_cache']
    records = cache_manifest(cache, cohort, C['patch'], C['fs'])
    if cohort == 'uci395':
        records = records.loc[records.task == C['uci_primary_task']].copy()
    records = records.rename(columns={'idx': 'cache_idx'}).reset_index(drop=True)
    records['idx'] = np.arange(len(records))
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(splitter.split(np.zeros(len(records)), records.label, records.subject))
    folds = np.full(len(records), -1, int)
    for fold, (train, test) in enumerate(splits):
        if set(records.subject.iloc[train]) & set(records.subject.iloc[test]):
            raise AlignmentError('subject leakage in fold')
        if set(records.label.iloc[test]) != {0, 1}:
            raise AlignmentError('held-out fold lacks a class; no automatic split change')
        folds[test] = fold
    if (folds < 0).any(): raise AlignmentError('incomplete folds')
    records['fold'] = folds
    if cohort == 'newhandpd':
        old = pd.read_csv(ROOT/C['discovery_run']/'record_manifest.csv')
        if len(old) != len(records) or not np.array_equal(old.record_id, records.record_id):
            raise AlignmentError('NewHandPD records differ from discovery bank')
        field = 'fold_actual' if 'fold_actual' in old else 'fold'
        if not np.array_equal(old[field].to_numpy(int), folds):
            raise AlignmentError('NewHandPD fold identities differ')
        if records.subject.nunique() != 66 or len(records) != 264:
            raise AlignmentError('NewHandPD bank denominator differs')
    return records, splits


FAST_PLAN=ROOT/'plans/attention_audit_replication20_fast_20261020'
EXECUTION_LOCK=json.loads((FAST_PLAN/'execution_lock.json').read_text())
EXECUTION_ID=EXECUTION_LOCK['execution_id']
if EXECUTION_ID!=spec_digest({k:v for k,v in EXECUTION_LOCK.items() if k!='execution_id'}):
    raise AlignmentError('execution lock digest mismatch')
if EXECUTION_LOCK['parent_design_id']!=LOCK['design_id']:
    raise AlignmentError('execution patch belongs to a different scientific design')
for relative,expected_hash in EXECUTION_LOCK['source_sha256'].items():
    if file_sha256(ROOT/relative)!=expected_hash:
        raise AlignmentError('versioned execution source changed: '+relative)
FAST_ROOT=ROOT/'results/attention_replication20_fast'

manifest={}

In [ ]:
tables=[];inputs=[]
for cohort in ('newhandpd','uci395'):
    pointer=FAST_ROOT/cohort/'latest_complete.json'
    if not pointer.is_file():raise FileNotFoundError('dataset has no complete run: '+str(pointer))
    ref=json.loads(pointer.read_text());directory=ROOT/ref['path']
    manifest_path=directory/'analysis_manifest.json'
    if file_sha256(manifest_path)!=ref['manifest_sha256']:raise AlignmentError('completed manifest changed')
    info=json.loads(manifest_path.read_text())
    if (info['completion_status']!='complete' or info['design_id']!=LOCK['design_id']
            or info['protocol_sha256']!=LOCK['protocol_sha256'] or info['n_checkpoints']!=200):
        raise AlignmentError('incomplete/different replication bank: '+cohort)
    if info.get('execution_id')!=EXECUTION_ID or info.get('backend_validation_pending',True):
        raise AlignmentError('fast execution identity/validation incomplete: '+cohort)
    if file_sha256(directory/'fast_backend_profile.json')!=info['fast_backend_profile_sha256']:
        raise AlignmentError('fast backend profile changed: '+cohort)
    models=pd.read_csv(directory/'model_inventory.csv')
    if len(models)!=200 or models.duplicated(['architecture','seed','fold']).any():
        raise AlignmentError('checkpoint inventory is not unique and complete')
    for arm in C['architectures']:
        if set(models.loc[models.architecture==arm,'seed'])!=set(range(20)):
            raise AlignmentError('missing planned seed')
    source=directory/'primary_mean_D_unadjusted.csv'
    for name in ('primary_mean_D_unadjusted.csv','model_inventory.csv','alignment_checks.csv','swap_pair_curves.csv'):
        if file_sha256(directory/name)!=info['result_sha256'][name]:raise AlignmentError('result file changed: '+name)
    if not pd.read_csv(directory/'alignment_checks.csv').passed.all():raise AlignmentError('alignment check failed')
    result=pd.read_csv(source)
    if len(result)!=4 or set(result.cohort)!={cohort}:raise AlignmentError('primary stratum denominator')
    tables.append(result);inputs.append(dict(cohort=cohort,path=str(directory.relative_to(ROOT)),
        primary_sha256=file_sha256(source),runtime_id=info['runtime_id'],
        n_subjects=info['n_subjects'],n_records=info['n_records'],
        primary_training_seeds=info['primary_training_seeds']))
combined=pd.concat(tables,ignore_index=True)
if len(combined)!=8:raise AlignmentError('eight-test family incomplete')
values=combined.p_value.to_numpy(float)
# An undefined test occupies its place conservatively as p=1 in the fixed family;
# its own reported adjusted value remains undefined.
order=np.argsort(np.where(np.isfinite(values),values,1.))
adjusted=np.maximum.accumulate((8-np.arange(8))*np.where(np.isfinite(values[order]),values[order],1.))
mapped=np.empty(8,float);mapped[order]=np.minimum(adjusted,1.)
combined['p_holm_eight_primary_tests']=np.where(np.isfinite(values),mapped,np.nan)
combined['ci_multiplicity']='nominal percentile; not simultaneous intervals'
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=FAST_ROOT/'combined'/LOCK['design_id']/EXECUTION_ID/stamp
OUT.mkdir(parents=True,exist_ok=False)
atomic_csv(OUT/'primary_eight_strata_holm.csv',combined)
atomic_json(OUT/'analysis_manifest.json',dict(completion_status='complete',
    design_id=LOCK['design_id'],execution_id=EXECUTION_ID,inputs=inputs,family_size=8,
    endpoint='subject-balanced own minus transferred RMS-control advantage',
    uncertainty='conditional subject intervals; no seed-population interval',
    public_preregistration=False,old_preregistration_changed=False,
    training_performed=False,model_inference_performed=False,new_interventions_performed=False))
display(combined)
print('COMPLETE:',OUT)
print('No cross-dataset pooling of raw logit-AOPC; each row remains a separate stratum.')